# Train Real ML Model on CICIDS2017 DDoS Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1. Load the real dataset
print("Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...")
df = pd.read_csv('../Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')

# Clean column names
df.columns = df.columns.str.strip()
print("Dataset shape:", df.shape)


Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...


Dataset shape: (225745, 79)


In [2]:
# 2. Select Features matching the Extractor
# We will use Flow Bytes/s, Flow Packets/s, Total Fwd Packets, Total Length of Fwd Packets
# as they map well to what we calculate real-time.
features_to_use = ['Flow Bytes/s', 'Flow Packets/s']

df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=features_to_use + ['Label'])

X = df_clean[features_to_use]
# Convert labels -> 1 for BENIGN, -1 for Attack (DDoS etc.)
y = df_clean['Label'].apply(lambda x: 1 if x == "BENIGN" else -1)


In [3]:
# 3. Train Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Random Forest model on REAL data...")
model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))


Training Random Forest model on REAL data...


Accuracy: 90.17%
              precision    recall  f1-score   support

          -1       0.91      0.92      0.91     25724
           1       0.89      0.88      0.88     19419

    accuracy                           0.90     45143
   macro avg       0.90      0.90      0.90     45143
weighted avg       0.90      0.90      0.90     45143



In [4]:
# 4. Save the new powerful model
joblib.dump(model, 'model.joblib')
print("Saved REAL model to model.joblib")


Saved REAL model to model.joblib
